# FINAL CODE

In [6]:
import re
import pandas as pd

# Overrides: more robust item header parsing and style detection

HEADER_PAT_A = re.compile(r"^([A-Z0-9\-]{3,})\s+(?:([YWR]G)\s*?750|([YWR]G750))?\s*(\d{8})\s+(STA|\d+)\s+(\d+)\s+(\d+)\s+Piece\b")
HEADER_PAT_B = re.compile(r"^([A-Z][A-Z0-9\-]{2,})\s+(STA|\d+)\s+(\d+)\s+(\d+)\s+Piece\b")
HEADER_PAT_C = re.compile(r"^(\d{8})\s+(STA|\d+)\s+(\d+)\s+(\d+)\s+Piece\b")
TOTAL_SKU_PAT = re.compile(r"^TOTAL\s+(\d{8})\b")

STYLE_TOKEN_PAT = re.compile(r"\b([A-Z][A-Z\-]*\d{3,})\b")
EXCLUDE_STYLE_TOKENS = {"YG750", "WG750", "RG750", "YG", "WG", "RG"}


def is_item_header_v2(line: str) -> bool:
    return bool(HEADER_PAT_A.search(line) or HEADER_PAT_B.search(line) or HEADER_PAT_C.search(line))


def find_style_in_block(block_lines: list[str]) -> str:
    for ln in block_lines[1:]:  # skip the header itself
        # skip obvious non-style lines
        if ln.upper().startswith("TOTAL "):
            break
        m = STYLE_TOKEN_PAT.search(ln)
        if m:
            token = m.group(1)
            if token not in EXCLUDE_STYLE_TOKENS and not token.isdigit() and len(token) >= 5:
                return token
    return ""


def parse_items_v2(text: str) -> list[dict]:
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    items: list[dict] = []
    i = 0
    while i < len(lines):
        line = lines[i]
        if not is_item_header_v2(line):
            i += 1
            continue

        style_code = ""
        sku_no = ""
        size_token = ""
        order_qty = ""

        mA = HEADER_PAT_A.search(line)
        mB = HEADER_PAT_B.search(line) if not mA else None
        mC = HEADER_PAT_C.search(line) if not (mA or mB) else None

        if mA:
            style_code = mA.group(1)
            # tone hint present implicitly via optional groups 2/3; we detect tone later from block
            sku_no = mA.group(4)
            size_token = mA.group(5)
            order_qty = mA.group(6)  # first qty
        elif mB:
            style_code = mB.group(1)
            size_token = mB.group(2)
            order_qty = mB.group(3)
            # sku to be found from TOTAL in block
        elif mC:
            # SKU-first header, style comes in following lines
            sku_no = mC.group(1)
            size_token = mC.group(2)
            order_qty = mC.group(3)
        else:
            i += 1
            continue

        # Collect block lines until next header or end; include TOTAL line
        block_lines = [line]
        j = i + 1
        sku_from_total = ""
        while j < len(lines):
            nxt = lines[j]
            if is_item_header_v2(nxt):
                break
            block_lines.append(nxt)
            t = TOTAL_SKU_PAT.search(nxt)
            if t:
                sku_from_total = t.group(1)
            j += 1
        i = j

        if not sku_no:
            sku_no = sku_from_total

        # If style not known (e.g., SKU-first case), try to find it within the block
        if not style_code:
            style_code = find_style_in_block(block_lines)

        # Tone detection: prefer explicit YG/WG/RG on header or block
        tone = ''
        joined = " ".join(block_lines)
        if re.search(r"\bWG\s*750|WG750", joined):
            tone = 'W'
        elif re.search(r"\bRG\s*750|RG750", joined):
            tone = 'R'
        elif re.search(r"\bYG\s*750|YG750", joined):
            tone = 'Y'
        else:
            if re.search(r"WHITE", joined, re.IGNORECASE):
                tone = 'W'
            elif re.search(r"ROSE", joined, re.IGNORECASE):
                tone = 'R'
            elif re.search(r"YELLOW", joined, re.IGNORECASE):
                tone = 'Y'

        # Fineness detection
        fineness = '750' if (re.search(r"\b18\s*CARA?\b", joined, re.IGNORECASE) or re.search(r"\b750\b", joined)) else ''

        # Diamond quality
        diamond_quality = ''
        for bl in block_lines:
            m = re.search(r"\b([A-Z]{1,2}-?SI\d|[A-Z]{1,2}-?VS\d?|[A-Z]{1,2}-?VVS\d?|[A-Z]{1,2}-?I\d)\b", bl)
            if m:
                diamond_quality = m.group(1)
                break

        # Carat line
        carat_line = ''
        for bl in block_lines:
            m = re.search(r"\b18\s*CARA?\s*-\s*750\b", bl, re.IGNORECASE)
            if m:
                carat_line = m.group(0)
                break
        if not carat_line and fineness == '750':
            carat_line = '18 CARA - 750'

        # Build fields
        # ItemSize only for Stylecodes starting with 'R'
        if style_code.startswith('R'):
            item_size = "" if size_token == 'STA' else (f"EU{size_token}" if size_token else "")
        else:
            item_size = ""

        metal = f"G{fineness}{tone}" if fineness and tone else (f"G{fineness}" if fineness else "")

        # Inputs
        priority = input(f"Enter Priority for SKU {sku_no or style_code}: ").strip()
        stamp_var = input(f"Enter stamp variable for SKU {sku_no or style_code} ('lgd' or leave blank for natural): ").strip().lower()
        stamp_variable_text = 'lgd' if stamp_var == 'lgd' else ''

        # Special remarks
        tone_to_desc = {'Y': 'YELLOW GOLD', 'W': 'WHITE GOLD', 'R': 'ROSE GOLD'}
        tone_desc = tone_to_desc.get(tone, '')
        parts = []
        if sku_no:
            parts.append(sku_no)
        if fineness or tone_desc:
            txt = " ".join([p for p in [fineness, tone_desc] if p]).strip()
            if txt:
                parts.append(txt)
        if diamond_quality:
            parts.append(f"DIA QLTY: {diamond_quality}")
        special_remarks = ",".join(parts)

        common_sentence = "Polishing and setting must be very well done."
        customer_prod_instruction = f"{carat_line}, {common_sentence}" if carat_line else common_sentence
        design_prod_instruction = "white rodium" if tone == 'W' else "no rodoium"

        items.append({
            'Sr.No': len(items) + 1,
            'Stylecode': style_code,
            'ItemSize': item_size,
            'OrderQty': order_qty,
            'OrderItemPcs': 1,
            'Metal': metal,
            'Tone': tone,
            'ItemPoNo': '',
            'ItemRefNo': '',
            'StockType': '',
            'Priority': priority,
            'MakeType': '',
            'CustomerProductionInstruction': customer_prod_instruction,
            'SpecialRemarks': special_remarks,
            'DesignProductionInstruction': design_prod_instruction,
            'StampInstruction': f"750+customer logo+{stamp_variable_text}".rstrip('+'),
            'OrderGroup': '',
            'Certificate': '',
            'SKUNo': sku_no,
            'Basestoneminwt': '',
            'Basestonemaxwt': '',
            'Basemetalminwt': '',
            'Basemetalmaxwt': '',
            'Productiondeliverydate': '',
            'Expecteddeliverydate': '',
            'SetPrice': '',
            'StoneQuality': '',
        })

    return items

# Re-run with the improved parser while keeping earlier utilities
try:
    pdf_path = pdf_file_path if 'pdf_file_path' in globals() else r'C:\Users\Pratik Mali\Desktop\tools\OrderProcessingTool\FSA\Fredy Sadik (FSA)_PO.pdf'
    text = extract_full_text_from_pdf(pdf_path)
    po_no = find_po_number(text)
    parsed_items = parse_items_v2(text)
    for it in parsed_items:
        it['ItemPoNo'] = po_no

    requested_columns = [
        'Sr.No','Stylecode','ItemSize','OrderQty','OrderItemPcs','Metal','Tone','ItemPoNo','ItemRefNo','StockType','Priority','MakeType','CustomerProductionInstruction','SpecialRemarks','DesignProductionInstruction','StampInstruction','OrderGroup','Certificate','SKUNo','Basestoneminwt','Basestonemaxwt','Basemetalminwt','Basemetalmaxwt','Productiondeliverydate','Expecteddeliverydate','SetPrice','StoneQuality'
    ]
    df_items = pd.DataFrame(parsed_items)
    for col in requested_columns:
        if col not in df_items.columns:
            df_items[col] = ''
    df_items = df_items[requested_columns]

    print("Mapped Items Data (improved parser):")
    print(df_items)

    out_path = pdf_path.rsplit('\\', 1)[0] + '\\FSA_mapped_items_5.xlsx'
    df_items.to_excel(out_path, indaex=False)
    print(f"Saved to: {out_path}")
except Exception as e:
    print(f"Error in improved mapping flow: {e}")


Mapped Items Data (improved parser):
   Sr.No   Stylecode ItemSize OrderQty  OrderItemPcs  Metal Tone  ItemPoNo  \
0      1  ER0000564A                 3             1  G750Y    Y  16047304   
1      2  PD0000453A                 3             1  G750Y    Y  16047304   
2      3    ER016388                25             1   G750       16047304   
3      4     PD01953                25             1  G750Y    Y  16047304   
4      5    E-115356                 5             1  G750Y    Y  16047304   

  ItemRefNo StockType  ... Certificate     SKUNo Basestoneminwt  \
0                      ...              10210233                  
1                      ...              10210234                  
2                      ...              10212596                  
3                      ...              10212597                  
4                      ...              10212611                  

  Basestonemaxwt Basemetalminwt Basemetalmaxwt Productiondeliverydate  \
0                 